# Section 1 — Single Agents

**Microsoft Agent Framework + Azure AI Foundry Workshop Athora**

[Microsoft Agent Framework](https://github.com/microsoft/agent-framework) is an open-source SDK for building AI agents and multi-agent workflows in Python and .NET. It unifies and extends the ideas in [Semantic Kernel](https://github.com/microsoft/semantic-kernel) and [AutoGen](https://github.com/microsoft/autogen) and is built by the same teams.

Agent Framework offers two primary categories of capabilities:

- **AI agents** — individual agents that use LLMs to process input, call tools and MCP servers, and produce responses. Backed by Azure OpenAI, OpenAI, or **Azure AI Foundry** model inference.
- **Workflows** — graph-based orchestrations that connect multiple agents and functions for complex, multi-step tasks (Sections 3 and 4).

Useful references that accompany this notebook:

- Framework overview: [`learn.microsoft.com/agent-framework/overview`](https://learn.microsoft.com/en-us/agent-framework/overview/agent-framework-overview)

- Foundry chat client: [`agent-framework-foundry`](https://github.com/microsoft/agent-framework/tree/main/python/packages/foundry)

## The story we are building today: PolicyPal

Imagine a customer-service representative picking up a call from a policyholder at an insurance provider. The customer is asking about their pension plan:
*“What’s the current status of my policy? Can I increase my contributions? And when could I realistically retire?”*
To answer these questions today, the representative typically has to navigate multiple internal systems, switching between policy data, contribution history, projections tools, and compliance guidelines. This slows things down and increases the risk of incomplete or inconsistent answers.

By the end of the day you will have built **PolicyPal**, an internal assistant designed to support customer-service representatives in real time. It handles data, projections, customer's profile, hands off to specialist agents (claims, risk, compliance), is **observable**, and can be **evaluated** for quality and safety before it touches real customer cases.

In this section we build PolicyPal v1: a single agent that can talk, use tools, hold a conversation, and remember things.

## Agents

All chat-style agents in Agent Framework are instances of the [`Agent`](https://github.com/microsoft/agent-framework/blob/main/python/packages/core/agent_framework/_agents.py) class. An agent is a thin coordinator around three things:

| Concept | Provided by |
|---------|-------------|
| **Chat client** | The thing that talks to the model. Examples: `FoundryChatClient`, `AzureOpenAIChatClient`, `OpenAIChatClient`. |
| **Instructions** | The system prompt — the agent's persona and rules. |
| **Tools** *(optional)* | Python functions the model can call to act on the world. |

Agents support out of the box:

1. Function calling (your own tools)
2. Multi-turn conversations via **sessions** (local or service-managed history)
3. Service-provided tools (MCP, code interpreter, file search, …)
4. Structured output
5. Streaming responses

All other agent types (Foundry hosted agent, OpenAI Assistants, …) are constructed differently but **run** identically. In this example we use the local `Agent` + `FoundryChatClient` combination throughout: the agent loop runs in our process and only calls Foundry for model inference, which avoids any dependency on the hosted Foundry Agent Service.

## Setup environment and credentials

We read two values from `.env`:

```env
FOUNDRY_PROJECT_ENDPOINT=https://<shared-project>.services.ai.azure.com/api/projects/<project>
FOUNDRY_MODEL=gpt-4.1
```

Authentication uses [`AzureCliCredential`](https://learn.microsoft.com/python/api/azure-identity/azure.identity.azureclicredential): the SDK reuses your last `az login`, so no keys live in the notebook. If anything fails in the next cell, raise your hand.

In [ ]:
%pip install -r requirements.txt -q

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model = os.environ["FOUNDRY_MODEL"]
print(f"Foundry endpoint: {endpoint}")
print(f"Model:            {model}")

### Imports

- `Agent` — the agent class.
- `FoundryChatClient` — chat client that calls a model deployment in your Foundry project.
- `AzureCliCredential` — token provider backed by `az login`.

These three imports cover roughly 90% of single-agent code you will write against Foundry.

In [ ]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print("agent-framework imports OK")

## Creating a chat client

The chat client is independent of any agent. You construct it once and can share it across multiple agents.

```python
client = FoundryChatClient(
    project_endpoint=...,  # Foundry project endpoint
    model=...,             # model deployment name
    credential=...,        # AzureCliCredential, ManagedIdentityCredential, ...
)
```

See the canonical sample: [`01_hello_agent.py`](https://github.com/microsoft/agent-framework/blob/main/python/samples/01-get-started/01_hello_agent.py).

In [ ]:
credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=credential,
)

## Creating our first agent: PolicyPal

To create an agent, construct an `Agent` with a chat client and instructions:

```python
Agent(
    client=client,
    name="PolicyPal",
    instructions="You are PolicyPal, an internal assistant for Athora Netherlands reps.",
)
```

Instructions are the cheapest, fastest way to shape behaviour. Same model + same question + different instructions = different answer.

In [ ]:
policypal = Agent(
    client=client,
    name="PolicyPal",
    instructions=(
        "You are PolicyPal, an internal assistant for Athora Netherlands customer-service representatives. "
        "You help reps answer questions about life insurance and pension policies. "
        "Keep replies concise, factual, and never invent policy numbers, balances, or dates."
    ),
)

## Running the agent

Call `await agent.run(user_input)`. The return value is an `AgentResponse` whose `.text` property holds the final assistant text. The agent is **stateless by default** — every call is a fresh conversation unless you supply a session (next section).

`run` accepts keyword arguments to customise each call (e.g. `max_tokens`, `temperature`, additional `tools`). Defaults come from the agent itself.

In [ ]:
response = await policypal.run(
    "In one sentence, introduce yourself to a new rep joining the team today."
)

print(response.text)

### Streaming responses

For chat-style UX you want tokens as they are generated. Pass `stream=True` to `agent.run(...)` and iterate over the chunks. Each chunk's `.text` holds a partial piece of the model's output.


In [ ]:
print("PolicyPal (streaming, one chunk per line):")
async for chunk in policypal.run(
    "In two  sentences, explain to a new rep what 'unit-linked life insurance' is.",
    stream=True,
):
    if chunk.text:
        print(repr(chunk.text), flush=True)


## Sessions and Multi-Turn Conversations

The agent above is stateless. If you ask a follow-up, it has no reference to the prior turn.

An [`AgentSession`](https://github.com/microsoft/agent-framework/blob/main/python/samples/01-get-started/03_multi_turn.py) holds the message history (and any session state) for one conversation. Create one with `agent.create_session()` and pass it to `run` via `session=...`. The same agent can drive many independent sessions in parallel.

If you do not pass a session, the framework uses a throwaway one that is discarded after the call.

In [ ]:
session = policypal.create_session()

rep_messages = [
    "I have Jan de Vries on the line — policy NL-2031-887.",
    "Can you tell me what kind of product that is, in one sentence?",
    "Remind me — what was the holder's name again?",
]

for msg in rep_messages:
    print(f"*** Rep:       {msg}")
    response = await policypal.run(msg, session=session)
    print(f"*** PolicyPal: {response.text}\n")

Notice that turn 3 successfully recalls the holder's name from turn 1 — only because the **same `session` object** was passed to all three calls. Drop `session=session` and the agent will not remember anything between turns. Try it!

### One agent, multiple independent conversations

The same `Agent` object can drive many sessions in parallel — useful when one PolicyPal instance serves many reps simultaneously. Each session is an isolated history; the agent itself stays stateless.

In [ ]:
session_a = policypal.create_session()
session_b = policypal.create_session()

ra1 = await policypal.run("Caller A is asking about policy NL-2031-887.", session=session_a)
rb1 = await policypal.run("Caller B is asking about policy NL-4408-552.", session=session_b)
print("Session A turn 1:", ra1.text, "\n")
print("Session B turn 1:", rb1.text, "\n")

ra2 = await policypal.run("Which policy number were we discussing again?", session=session_a)
rb2 = await policypal.run("Which policy number were we discussing again?", session=session_b)
print("Session A turn 2:", ra2.text)
print("Session B turn 2:", rb2.text)

### Section References

To learn more about agent runs, streaming responses and message types, refer to the following:

1. [Running Agents](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/?pivots=programming-language-python) - Everything we've just covered and more!

2. [Documentation on agent_framework package](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework?view=agent-framework-python-latest) - We highly recommend you to read about different agent response type class implementations (e.g. `AgentResponse` vs `AgentResponseUpdate`), will be very useful later on:\)

[Conversations & Memory Overview](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/?pivots=programming-language-python) - Additional Guidance

Code Samples:
 - [Basic Multi-Turn Conversation](https://github.com/microsoft/agent-framework/blob/main/python/samples/01-get-started/03_multi_turn.py)
 - [Agent Memory with context providers and session state](https://github.com/microsoft/agent-framework/blob/main/python/samples/01-get-started/04_memory.py)

## Function tools

An agent without tools can only talk. With tools it can **act** — query the policy admin system, run a pension projection, look up a claim, send a follow-up email.

In Agent Framework a tool is just a Python function. The framework reads the **type hints** and **docstring** to build the JSON schema sent to the model. The model decides when to call the tool, the framework executes it, and the result is fed back into the conversation. You write the function — the framework runs the loop.

The `@tool` decorator marks a function as a tool and lets you set runtime properties such as `approval_mode` (whether the framework should ask for human approval before invocation). For workshop brevity we set `approval_mode="never_require"`. 

### Defining a tool

Tooling support may vary between different agent types. Some agents may allow developers to customize the agent at construction time by providing external function tools or by choosing to activate specific built-in tools that are supported by the agent. 

The `Agent` is an agent class that can be used to build agentic capabilities on top of any inference service. It comes with support for:

1. Using your own function tools with the agent
2. Using built-in tools that the underlying service may support
3. Using hosted tools like web search and MCP (Model Context Protocol) servers

In the following example, we define a simple tool that retrieves information about a customer's policy. If provided policy_number exists in _POLICY_BOOK, it will return the policy data, if not - return the error message.

In [ ]:
from typing import Annotated
from agent_framework import tool
from pydantic import Field

# Mock policy book — in production this would call the real backend.
_POLICY_BOOK = {
    "NL-2031-887": {
        "holder": "Jan de Vries",
        "product": "Pension - defined contribution",
        "balance_eur": 78_420.55,
        "monthly_contribution_eur": 350.00,
        "inception_date": "2014-06-01",
    },
    "NL-4408-552": {
        "holder": "Sanne Bakker",
        "product": "Life insurance - term",
        "balance_eur": 0.0,
        "monthly_contribution_eur": 42.50,
        "inception_date": "2019-11-15",
    },
}

@tool(approval_mode="never_require")
def get_policy(
    policy_number: Annotated[str, Field(description="Athora policy number, e.g. NL-2031-887.")],
) -> dict:
    """Look up a life insurance or pension policy by its policy number.

    Returns key fields: holder, product, balance_eur, monthly_contribution_eur, inception_date.
    """
    return _POLICY_BOOK.get(policy_number, {"error": f"No policy found for {policy_number}"})

### Giving the tool to PolicyPal

Tools are passed as a list to the `Agent` constructor (agent-level tools available on every run) or to `agent.run(..., tools=[...])` (run-level tools available for that call only).

In [ ]:
policypal_with_lookup = Agent(
    client=client,
    name="PolicyPal",
    instructions=(
        "You are PolicyPal, an internal assistant for Athora Netherlands reps. "
        "Use the get_policy tool to look up real policy data. "
        "Never guess at policy numbers, balances, or dates — if a tool returns an error, say so plainly."
    ),
    tools=[get_policy],
)

result = await policypal_with_lookup.run(
    "A rep is on the phone with the holder of NL-2031-887. What product do they have?"
)
print(result.text)

Behind the scenes the framework ran a small loop:

1. Sent the rep's message + the `get_policy` schema to the model.
2. The model replied with a tool call: `get_policy(policy_number="NL-2031-887")`.
3. The framework ran the Python function and captured the return value.
4. The framework sent the tool result back to the model.
5. The model produced the final natural-language answer.

You will *see* this loop visually in **Section 2 about DevUI**.

### Multiple tools

Real agents need more than one tool. Add as many as you like - the model picks which to call (and may chain several). Below we add a pension projection tool. Then we ask a question that should make PolicyPal call **both** tools in a single turn.

In [ ]:
@tool(approval_mode="never_require")
def project_pension_balance(
    policy_number: Annotated[str, Field(description="Pension policy number.")],
    years: Annotated[int, Field(description="How many years to project forward.")],
) -> str:
    """Project the pension policy balance `years` years from today (4% annual return assumption)."""
    p = _POLICY_BOOK.get(policy_number)
    if not p or p.get("balance_eur", 0) == 0:
        return f"No projectable pension found for {policy_number}."
    rate = 0.04
    balance = p["balance_eur"]
    monthly = p["monthly_contribution_eur"]
    future = balance * (1 + rate) ** years + monthly * 12 * years * (1 + rate / 2)
    return f"Projected balance for {policy_number} in {years} years: ~ EUR {future:,.0f} (illustrative)."

policypal_full = Agent(
    client=client,
    name="PolicyPal",
    instructions=(
        "You are PolicyPal. Always use tools for policy facts and projections. "
        "Make clear that projections are illustrative."
    ),
    tools=[get_policy, project_pension_balance],
)

result = await policypal_full.run(
    "Confirm the current balance of NL-2031-887 and project it 10 years out."
)
print(result.text)

And the `@tool` decorator can override the function name and description shown to the model:

```python
@tool(name="policy_lookup", description="Look up an Athora policy by its number.")
def get_policy(...):
    ...
```

More patterns (approval flows, hosted tools, MCP tools) are in [`python/samples/02-agents/tools`](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/tools).

Tools can also be defined as **methods on a class** (useful when several tools share state — e.g., a database connection or an HTTP session).
In the following exercise, we will guide you through an exercise that implements exactly that.


### Exercise 1: Implement an Agent with Function Tools

Open **`01-exercises.ipynb`** and complete the exercise 1. You will build a `ClaimsTracker` class where multiple tool methods share an in-memory claims list — the same pattern you would use to wrap a real database connection or API client.

## Agent Memory with Context Providers

A session keeps state for the duration of *one* conversation. But customers call back. The next time someone phones in, PolicyPal should already know his preferred language, his risk profile, and any compliance flags without the rep retyping it.

A [`ContextProvider`](https://github.com/microsoft/agent-framework/blob/main/python/samples/01-get-started/04_memory.py) is the framework hook for more persistent memory. Is it a structured pattern for loading the state, injecting information into prompt, observing the response and saving the new state.

The provider has two methods the framework calls for you on each run:

- `before_run` — extend the agent's instructions / context for this call (e.g. inject text into the system prompt on what we know about the customer).
- `after_run` — observe the call and update your storage (e.g. record what the customer just told you)


The provider is registered on the agent via `context_providers=[...]`. You can attach several; they run in order.

### A minimal CustomerProfileMemory

We back the provider with a plain Python dict - stand-in for what would be Cosmos DB / Azure SQL / Azure AI Search at Athora.

In [ ]:
from typing import Any
from agent_framework import AgentSession, ContextProvider, SessionContext

class CustomerProfileMemory(ContextProvider):
    """Inject known customer profile facts as extra instructions before each run."""

    DEFAULT_SOURCE_ID = "customer_profile"

    def __init__(self) -> None:
        super().__init__(self.DEFAULT_SOURCE_ID)
        self._facts: dict[str, str] = {}

    def remember(self, key: str, value: str) -> None:
        self._facts[key] = value

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        if not self._facts:
            return
        bullets = "\n".join(f"- {k}: {v}" for k, v in self._facts.items())
        context.extend_instructions(self.source_id, f"Known customer profile:\n{bullets}")

In [ ]:
profile = CustomerProfileMemory()
profile.remember("holder_name", "Jan de Vries")
profile.remember("preferred_language", "Dutch")
profile.remember("risk_profile", "conservative")
profile.remember("compliance_flag", "identity verified 2026-01-15")

policypal_with_memory = Agent(
    client=client,
    name="PolicyPal",
    instructions=(
        "You are PolicyPal. Always honour the known customer profile when advising reps. "
        "If the customer's risk profile is conservative, do not propose aggressive products."
    ),
    context_providers=[profile],
)

result = await policypal_with_memory.run(
    "The rep wants to suggest a way to grow this customer's pension faster. What can you propose?"
)
print(result.text)

Even though the rep never mentioned Dutch language or a conservative risk profile, the suggestion respects both - `CustomerProfileMemory.before_run` extended the agent's instructions with the profile bullets just before the model was called. If we want to preserve the context after the run (e.g. capture the agent response), we can do that with `after_run`.

 **Note:** The durability of context providers is in your storage layer, not in the model. It is your responsibility to manage this storage, but there are many options available (Cosmos DB, Azure AI Search, Redis, SQL, etc.). Keep in mind that at the end of the day you are still consuming tokens on every run, so as your stored facts grow you will need to summarize and retrieve selectively.

This same hook is how more advanced providers work: an Azure AI Search provider that injects relevant product disclosure pages, a Cosmos DB provider that injects last-call summaries, or a graph provider that injects related entities.

Learn more about context providers: 
- [User Documentation](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/context-providers?pivots=programming-language-python)
- [Context Provider Samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/context_providers)
- [Conversation and Session Management Samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/conversations)

### Exercise 2: Build a Memory Extraction Provider

Open **`01-exercises.ipynb`** and complete the exercise 2. You will build a `SimpleMemoryProvider` that uses both `before_run` and `after_run`, observes user messages for stated preferences, stores them, and injects them as context on future turns.

## Agent Middleware

Middleware in the Agent Framework provides a powerful way to intercept, modify, and enhance agent interactions at various stages of execution. You can use middleware to implement cross-cutting concerns such as logging, security validation, error handling, and result transformation without modifying your core agent or function logic.

Function-based middleware is the simplest way to implement middleware using async functions. This approach is ideal for stateless operations and provides a lightweight solution for common middleware scenarios.

1. Agent Middleware - uses [`AgentRunContext`](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.agentruncontext?view=agent-framework-python-latest)
2. Function Middleware - uses [`FunctionInovationContext`](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.functioninvocationcontext?view=agent-framework-python-latest)
3. Chat Middleware - uses [`ChatContext`](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.chatcontext?view=agent-framework-python-latest)

Each middleware type has its own context objects and type of information available from it. Learn more about usage and best practices in [the following user guide](https://learn.microsoft.com/en-us/agent-framework/agents/middleware/?pivots=programming-language-python). Make sure to check out the [developer docs](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.agentmiddleware?view=agent-framework-python-latest) to understand middleware class implementation.

The following example demonstrates how middleware can be used to log execution details at a function (logs when a tool function is called and completed) and agent (logs before and after the entire agent run) levels. This gives you visibility and control over the agent’s behavior.

In [ ]:
from datetime import datetime
from collections.abc import Callable, Awaitable

from agent_framework import Agent, agent_middleware, FunctionInvocationContext
from agent_framework.openai import OpenAIChatClient  # or FoundryChatClient


def get_time() -> str:
    """Get the current time."""
    return datetime.now().strftime("%H:%M:%S")


async def logging_function_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    print(f"[Function] Calling {context.function.name}")
    print(f"[Function] Arguments: {context.arguments}")

    await next()

    print(f"[Function] {context.function.name} completed")
    print(f"[Function] Result: {context.result}")


@agent_middleware
async def logging_agent_middleware(context, next):
    print("[Agent] BEFORE execution")

    await next() 

    print(f"[Agent] AFTER execution | Agent output: {context.result}")


agent = Agent(
    client=client,
    name="TimeAgent",
    instructions="You can tell the current time.",
    tools=[get_time],
    middleware=[
        logging_agent_middleware,     # wraps entire run
        logging_function_middleware,  # wraps tool calls only
    ],
)

result = await agent.run("What time is it?")
print("Run1 response:", result.text)

result = await agent.run("Can you help?")
print("Run2 response:", result.text)

### Exercise 3: Implement Agent Middleware

 Open **`01-exercises.ipynb`** and complete the exercise 3. You will implement three middleware functions that log what happens inside the agent at each layer — tool calls, LLM requests, and overall execution time.

  Checkout more middleware code samples:

   - [Official middleware samples](https://github.com/microsoft/agent-framework/blob/main/python/samples/02-agents/middleware/README.md)
   - [Understanding middleware scopes](https://learn.microsoft.com/en-us/agent-framework/agents/middleware/agent-vs-run-scope?pivots=programming-language-python)

## Recap and what's next

You can now:

- Construct a `FoundryChatClient` and an `Agent` with instructions.
- Run an agent (non-streaming and streaming) and read `response.text`.
- Hold multi-turn conversations with `agent.create_session()` and `session=...`.
- Run multiple independent sessions on the same agent.
- Equip an agent with `@tool`-decorated Python functions and let the model orchestrate calls.
- Inject durable memory through a `ContextProvider`.

**Next: Section 2 - Foundry-Backed Agents + DevUI.** We take this same PolicyPal and explore it in DevUI to *see* the tool-call loop, message history, and timing — same agent, no code changes.

**Further reading:**

- All `01-get-started` samples: [`microsoft/agent-framework/python/samples/01-get-started`](https://github.com/microsoft/agent-framework/tree/main/python/samples/01-get-started)
- Agent types overview: [Agent types](https://learn.microsoft.com/en-us/agent-framework/user-guide/agents/agent-types/?pivots=programming-language-python)
- Tools and approvals: [`python/samples/02-agents/tools`](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/tools)
- Agent architecture: [Agent pipeline](https://learn.microsoft.com/en-us/agent-framework/agents/agent-pipeline?pivots=programming-language-python)
